# Google Earth Engine con Python para analizar vegetación

En esta práctica configuraremos Earth Engine en Google Colab, cargaremos un GeoPackage de frutales o de uso del suelo y analizaremos imágenes Sentinel-2. El recorrido incluye composiciones RGB y falso color, estructura matricial, niveles digitales, histogramas, índices espectrales, series temporales y correlaciones.

**Regla de trabajo:** ejecute una celda por vez. Las decisiones se toman editando las líneas indicadas en el código; no se utilizan botones ni selectores. No continúe si el resultado esperado no aparece.

## Propósito y resultados de aprendizaje

Al finalizar la clase podrá:

- configurar Google Colab y autenticar Earth Engine;
- elegir mediante líneas de código una base de frutales o de uso del suelo;
- aplicar filtros espaciales, temporales y de nubosidad a Sentinel-2;
- relacionar una matriz de niveles digitales con los píxeles de una banda;
- comparar color verdadero, falso color, TOA y reflectancia de superficie;
- interpretar histogramas y mapas de índices espectrales;
- extraer y comparar series temporales de varios índices;
- evaluar relaciones entre índices mediante correlación;
- formular interpretaciones prudentes sobre vigor, fenología, humedad y posible estrés.

**Producto de la sesión:** cuaderno ejecutado con mapas comparativos, gráficos interpretados, series temporales y una matriz de correlación.

## Preparación antes de ejecutar

Necesita una cuenta Google con acceso aprobado a Earth Engine, el identificador de un proyecto de Google Cloud habilitado y uno de los GeoPackage incluidos en `03_DATOS`.

- Ruta Frutales: `FRUTALES_VALPARAISO.gpkg`.
- Ruta Uso del suelo: `USOS_CASABLANCA.gpkg`.

La ruta mínima no requiere instalar Python en el computador. Colab ejecuta el código en una sesión temporal. Si la sesión se reinicia, vuelva a ejecutar las celdas desde el comienzo.

## 1 Preparar Colab

Esta celda instala geemap y las bibliotecas geoespaciales que usaremos. La instalación puede tardar uno o dos minutos.

In [ ]:
%pip install -q geemap geopandas pyogrio

In [ ]:
import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display

# Estilo común para todos los gráficos del cuaderno
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'legend.frameon': False,
})

COLORES_INDICES = {
    'NDVI': '#386641',
    'NDMI': '#277da1',
    'MSI': '#d97706',
    'NMDI': '#7b2cbf',
    'DWSI': '#c1121f',
}

print('Bibliotecas cargadas correctamente')

## 2 Autenticar e inicializar Earth Engine

Sustituya el texto por el identificador de su proyecto de Google Cloud. El proyecto debe estar habilitado para Earth Engine. La autenticación se repite cuando Colab reinicia la sesión.

In [ ]:
PROJECT_ID = 'lucasvituri'

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine conectado correctamente')

### Punto de control 1

Debe aparecer `Earth Engine conectado correctamente`. Si aparece un error de proyecto, revise el identificador y confirme que la API de Earth Engine está habilitada.

## 3 Elegir la base de análisis y cargar un GeoPackage

Primero elija si trabajará con especies frutales o con categorías de uso del suelo. Para frutales cargue `FRUTALES_VALPARAISO.gpkg`. Para uso del suelo cargue `USOS_CASABLANCA.gpkg`, una versión liviana preparada para la clase.

In [ ]:
from google.colab import drive
from pathlib import Path

# Montar Google Drive
drive.mount('/content/drive')


# Deje activa solamente una opción.
TIPO_ANALISIS = 'FRUTALES'
# TIPO_ANALISIS = 'USO_SUELO'

# Modifique esta ruta según la ubicación de la carpeta en su Drive
CARPETA_DRIVE = Path(
    '/content/drive/MyDrive/CLASE_01_GEE_PYTHON/03_DATOS'
)

archivo_esperado = (
    'FRUTALES_VALPARAISO.gpkg'
    if TIPO_ANALISIS == 'FRUTALES'
    else 'USOS_CASABLANCA.gpkg'
)

gpkg_path = CARPETA_DRIVE / archivo_esperado

if not gpkg_path.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo:\n{gpkg_path}\n\n'
        'Revise CARPETA_DRIVE y el nombre del archivo.'
    )

print('Base seleccionada:', TIPO_ANALISIS)
print('Archivo:', gpkg_path)

# Leer el GeoPackage
datos = gpd.read_file(gpkg_path)

if TIPO_ANALISIS == 'FRUTALES':
    campo_categoria = 'ESPECIE'
    campo_superficie = 'SUP_FRUTAL'
else:
    campo_categoria = 'USO_TIERRA'
    campo_superficie = 'SUPERF_HA'
    categoria_inicial = 'Terreno de Uso Agrícola'

datos[campo_categoria] = (
    datos[campo_categoria]
    .fillna('Sin categoría')
    .astype(str)
    .str.strip()
)

print('Filas:', len(datos))
print('CRS:', datos.crs)

display(
    datos[
        [campo_categoria, campo_superficie]
    ].head()
)

resumen_categorias = (
    datos.groupby(campo_categoria)[campo_superficie]
    .agg(
        poligonos='count',
        superficie='sum',
    )
    .sort_values('superficie', ascending=False)
)

display(resumen_categorias.head(15))

### Seleccionar una especie frutal o un uso del suelo

Use el selector para cambiar de categoría. En la ruta de frutales, `CEREZO` conecta directamente con Zambrano et al. (2025). En la ruta de uso del suelo puede explorar agricultura, bosque nativo, plantaciones, matorral y otras coberturas. Después de cambiar la categoría, vuelva a ejecutar desde la celda de selección hasta el análisis que quiera comparar.

In [ ]:
# MOSTRAR LAS CATEGORÍAS DISPONIBLES

opciones_categoria = sorted(
    datos[campo_categoria]
    .dropna()
    .unique()
    .tolist()
)

print('Categorías disponibles:')
for categoria in opciones_categoria:
    print('-', categoria)


In [ ]:
# Deje activa solamente una línea compatible con la base seleccionada.

# Ejemplos para la base FRUTALES
# CATEGORIA_ANALISIS = 'CEREZO'
# CATEGORIA_ANALISIS = 'PALTO'
# CATEGORIA_ANALISIS = 'NOGAL'
CATEGORIA_ANALISIS = 'VID DE MESA'

# Ejemplos para la base USO_SUELO
# CATEGORIA_ANALISIS = 'Terreno de Uso Agrícola'
# CATEGORIA_ANALISIS = 'Bosque Nativo Renoval Semidenso'
# CATEGORIA_ANALISIS = 'Plantación Adulta'
# CATEGORIA_ANALISIS = 'Rotación Cultivo-Pradera'

print('Categoría seleccionada:', CATEGORIA_ANALISIS)

In [ ]:
# VALIDAR LA CATEGORÍA SELECCIONADA

if CATEGORIA_ANALISIS not in opciones_categoria:
    raise ValueError(
        f'La categoría "{CATEGORIA_ANALISIS}" no está disponible.\n'
        'Revise la lista mostrada en la celda anterior.'
    )

# FILTRAR LOS POLÍGONOS DE LA CATEGORÍA

seleccion = datos[
    datos[campo_categoria].eq(CATEGORIA_ANALISIS)
].copy()

seleccion = seleccion.to_crs(epsg=4326)

# Seleccionar el polígono de mayor superficie
parcela = seleccion.nlargest(
    1,
    campo_superficie,
).copy()

print('Base:', TIPO_ANALISIS)
print('Categoría seleccionada:', CATEGORIA_ANALISIS)
print('Polígonos disponibles:', len(seleccion))

display(
    parcela[
        [campo_categoria, campo_superficie]
    ]
)

# CONVERTIR EL POLÍGONO A EARTH ENGINE

aoi_fc = geemap.gdf_to_ee(parcela)
aoi = aoi_fc.geometry()

print('Área de análisis creada correctamente')

## 4 Explorar el catálogo y filtrar Sentinel 2

Usaremos reflectancia de superficie armonizada. Aplicaremos tres filtros: área, fechas y nubosidad de escena. Después enmascararemos nubes y sombras por píxel con la banda SCL.

In [ ]:
FECHA_INICIO = '2022-10-01'
FECHA_FIN = '2025-05-01'
NUBOSIDAD_MAX = 30

s2_sin_mascara = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', NUBOSIDAD_MAX))
)

print('Imágenes después de los filtros:', s2_sin_mascara.size().getInfo())

In [ ]:
def preparar_s2(imagen):
    # Escalar reflectancias y enmascarar nubes, sombras, cirrus y nieve.
    scl = imagen.select('SCL')
    mascara_valida = (
        scl.neq(3)      # sombra de nube
        .And(scl.neq(8))   # nube de probabilidad media
        .And(scl.neq(9))   # nube de probabilidad alta
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # nieve o hielo
    )

    reflectancia = (
        imagen
        .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
        .multiply(0.0001)
        .updateMask(mascara_valida)
    )

    return ee.Image(
        reflectancia.copyProperties(
            imagen,
            ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE'],
        )
    )


s2 = s2_sin_mascara.map(preparar_s2)
compuesto = s2.median().clip(aoi)

print('Colección preparada:', s2.size().getInfo(), 'imágenes')

### Punto de control 2

La cantidad de imágenes debe ser mayor que cero. Si es cero, amplíe el periodo o aumente `NUBOSIDAD_MAX`. Observe que el filtro de escena y la máscara por píxel cumplen funciones distintas.

## 5 Color verdadero y falso color

La composición RGB usa las bandas roja, verde y azul, por lo que se parece a la visión humana. El falso color coloca el infrarrojo cercano en el canal rojo: la vegetación con mayor respuesta en NIR suele aparecer en tonos rojos intensos.

**Qué observar:** compare límites de parcelas, diferencias internas del dosel, suelo expuesto y zonas con tonos menos homogéneos. El color sugiere patrones; todavía no demuestra vigor o estrés.

In [ ]:
# Parámetros de visualización. Puede modificar min, max y gamma.
VIS_RGB = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.02,
    'max': 0.30,
    'gamma': 1.15,
}

VIS_FALSO_COLOR = {
    'bands': ['B8', 'B4', 'B3'],
    'min': 0.02,
    'max': 0.45,
    'gamma': 1.10,
}

ESTILO_PARCELA = {
    'color': 'white',
    'fillColor': '00000000',
    'width': 2,
}


def crear_mapa_comparacion(imagen, visualizacion, nombre):
    # Crear un panel sin solicitar teselas de OpenStreetMap.
    mapa_panel = geemap.Map(
        basemap={},
        layout=widgets.Layout(width='50%', height='480px'),
    )
    mapa_panel.centerObject(aoi, 15)
    mapa_panel.addLayer(imagen, visualizacion, nombre)
    mapa_panel.addLayer(
        aoi_fc.style(**ESTILO_PARCELA),
        {},
        'Parcela',
    )
    return mapa_panel


mapa_rgb = crear_mapa_comparacion(
    compuesto,
    VIS_RGB,
    'Color verdadero RGB',
)
mapa_falso_color = crear_mapa_comparacion(
    compuesto,
    VIS_FALSO_COLOR,
    'Falso color NIR',
)

# Los dos mapas mantienen el mismo centro y nivel de zoom.
widgets.jslink((mapa_rgb, 'center'), (mapa_falso_color, 'center'))
widgets.jslink((mapa_rgb, 'zoom'), (mapa_falso_color, 'zoom'))

display(
    widgets.HBox(
        [mapa_rgb, mapa_falso_color],
        layout=widgets.Layout(width='100%'),
    )
)

### Exploración de contraste para la Unidad 4

Los límites `min` y `max` determinan cómo la reflectancia se distribuye entre tonos oscuros y claros. Cambiar estos parámetros modifica la representación, pero no los valores originales.

**Qué observar en el mapa:** active una capa por vez y compare la separación visual entre copa, suelo y sombras. Un contraste muy estrecho puede exagerar diferencias pequeñas; uno muy amplio puede ocultarlas.

In [ ]:
mapa_contraste = geemap.Map(basemap={})
mapa_contraste.centerObject(aoi, 15)

CONTRASTES_RGB = {
    'Contraste estrecho': {'min': 0.05, 'max': 0.20},
    'Contraste medio': {'min': 0.02, 'max': 0.30},
    'Contraste amplio': {'min': 0.00, 'max': 0.50},
}

for posicion, (nombre, limites) in enumerate(CONTRASTES_RGB.items()):
    parametros = {
        'bands': ['B4', 'B3', 'B2'],
        **limites,
    }
    mapa_contraste.addLayer(
        compuesto,
        parametros,
        nombre,
        posicion == 0,
    )

mapa_contraste.addLayer(
    aoi_fc.style(**ESTILO_PARCELA),
    {},
    'Parcela',
)
mapa_contraste

### Comparar TOA y reflectancia de superficie

`COPERNICUS/S2_HARMONIZED` representa reflectancia en el tope de la atmósfera (TOA), mientras que `COPERNICUS/S2_SR_HARMONIZED` ofrece reflectancia de superficie corregida atmosféricamente. Se compara la media de la misma escena y área.

**Qué observar en el gráfico:** identifique las bandas con mayor diferencia entre TOA y SR. Esa separación muestra que la corrección atmosférica no afecta todas las longitudes de onda de la misma manera.

In [ ]:
imagen_sr_original = ee.Image(s2_sin_mascara.first())
indice_escena = ee.String(
    imagen_sr_original.get('system:index')
).getInfo()

toa_coincidente = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filter(ee.Filter.eq('system:index', indice_escena))
)

if toa_coincidente.size().getInfo() == 0:
    print(
        'No se encontró una escena TOA con el mismo índice. '
        'Continúe con la siguiente sección.'
    )
else:
    imagen_toa = ee.Image(toa_coincidente.first())
    bandas_comparacion = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

    valores_sr = (
        imagen_sr_original
        .select(bandas_comparacion)
        .multiply(0.0001)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=20,
            maxPixels=1e7,
        )
        .getInfo()
    )
    valores_toa = (
        imagen_toa
        .select(bandas_comparacion)
        .multiply(0.0001)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=20,
            maxPixels=1e7,
        )
        .getInfo()
    )

    comparacion_toa_sr = pd.DataFrame(
        {
            'TOA': [valores_toa.get(banda) for banda in bandas_comparacion],
            'Superficie': [valores_sr.get(banda) for banda in bandas_comparacion],
        },
        index=bandas_comparacion,
    )
    display(comparacion_toa_sr.round(3))

    ax = comparacion_toa_sr.plot(
        kind='bar',
        figsize=(9, 4),
        color=['#bc6c25', '#386641'],
    )
    ax.set(
        xlabel='Banda Sentinel-2',
        ylabel='Reflectancia media',
        title='Reflectancia TOA y de superficie',
    )
    plt.xticks(rotation=0)
    plt.show()

## 6 Píxeles y niveles digitales para la Unidad 4

Una banda satelital es una matriz: cada fila y columna localiza un píxel, y cada celda guarda un nivel digital. El parche siguiente usa B4 sin aplicar el factor de escala.

**Qué observar:** relacione la forma impresa `(filas, columnas)` con la cuadrícula coloreada. Tonos semejantes representan niveles digitales cercanos; valores diferentes pueden corresponder a copa, suelo, sombra o variación radiométrica.

In [ ]:
# La mediana amplía la cobertura y conserva los niveles digitales.
banda_b4 = (
    s2_sin_mascara
    .select('B4')
    .median()
    .rename('B4')
    .toInt32()
)

# Convertir una vecindad de 9 x 9 píxeles en una matriz por cada píxel central.
kernel = ee.Kernel.square(
    radius=4,
    units='pixels',
    normalize=False,
)

vecindad_b4 = banda_b4.neighborhoodToArray(
    kernel=kernel,
    defaultValue=-9999,
)

# Muestrear directamente la imagen de vecindades dentro de la parcela.
# Así se evita perder el dato al realizar una segunda consulta sobre un punto.
muestra_vecindad = ee.Feature(
    vecindad_b4.sample(
        region=aoi,
        scale=10,
        numPixels=1,
        seed=42,
        geometries=False,
        dropNulls=True,
    ).first()
)

matriz_lista = muestra_vecindad.get('B4').getInfo()

if matriz_lista is None:
    raise ValueError(
        'No se encontró una vecindad válida de B4 dentro de la parcela. '
        'Amplíe el periodo o aumente NUBOSIDAD_MAX.'
    )

matriz_b4 = np.asarray(matriz_lista, dtype=float)

if matriz_b4.ndim != 2 or matriz_b4.size == 0:
    raise ValueError(
        f'La extracción no produjo una matriz 2D. Forma obtenida: '
        f'{matriz_b4.shape}'
    )

mascara_valida = np.isfinite(matriz_b4) & (matriz_b4 != -9999)
cantidad_validos = int(mascara_valida.sum())

if cantidad_validos == 0:
    raise ValueError(
        'La matriz no contiene niveles digitales válidos de B4. '
        'Pruebe otra fecha o aumente NUBOSIDAD_MAX.'
    )

print('Forma de la matriz:', matriz_b4.shape)
print('Píxeles válidos:', cantidad_validos)
print('Niveles digitales de B4:')
print(np.where(mascara_valida, matriz_b4, np.nan))

# En el gráfico, el gris representa ausencia de datos.
matriz_visual = np.where(mascara_valida, matriz_b4, np.nan)
valores_validos = matriz_visual[mascara_valida]

limite_minimo = valores_validos.min()
limite_maximo = valores_validos.max()

if limite_minimo == limite_maximo:
    limite_minimo -= 1
    limite_maximo += 1

mapa_colores = plt.colormaps['Reds'].copy()
mapa_colores.set_bad('#d9d9d9')

fig, ax = plt.subplots(figsize=(7, 6))
imagen_matriz = ax.imshow(
    matriz_visual,
    cmap=mapa_colores,
    vmin=limite_minimo,
    vmax=limite_maximo,
)

# Escribir el nivel digital dentro de cada píxel válido.
umbral_texto = (limite_minimo + limite_maximo) / 2
for fila in range(matriz_visual.shape[0]):
    for columna in range(matriz_visual.shape[1]):
        valor = matriz_visual[fila, columna]
        if np.isfinite(valor):
            color_texto = 'white' if valor > umbral_texto else 'black'
            ax.text(
                columna,
                fila,
                f'{valor:.0f}',
                ha='center',
                va='center',
                fontsize=7,
                color=color_texto,
            )

fig.colorbar(
    imagen_matriz,
    ax=ax,
    label='Nivel digital B4',
    shrink=0.82,
)
ax.set(
    xlabel='Columna de la matriz',
    ylabel='Fila de la matriz',
    title='Vecindad de 9 × 9 píxeles — banda B4',
)
plt.show()


## 7 NDVI e histograma

El NDVI contrasta la reflectancia del infrarrojo cercano y del rojo. Valores mayores suelen acompañar una mayor presencia y actividad de vegetación verde; valores bajos pueden representar suelo, senescencia, sombra o escasa cobertura.

**Qué observar en los mapas:** compare el patrón RGB con el NDVI. Busque sectores internos consistentemente altos o bajos, pero no interprete un píxel aislado como estrés.

In [ ]:
ndvi = compuesto.normalizedDifference(['B8', 'B4']).rename('NDVI')

VIS_NDVI = {
    'min': 0,
    'max': 0.9,
    'palette': ['8c510a', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e'],
}

mapa_rgb_ndvi = crear_mapa_comparacion(
    compuesto,
    VIS_RGB,
    'Color verdadero RGB',
)
mapa_ndvi = crear_mapa_comparacion(
    ndvi,
    VIS_NDVI,
    'NDVI',
)

widgets.jslink((mapa_rgb_ndvi, 'center'), (mapa_ndvi, 'center'))
widgets.jslink((mapa_rgb_ndvi, 'zoom'), (mapa_ndvi, 'zoom'))

display(
    widgets.HBox(
        [mapa_rgb_ndvi, mapa_ndvi],
        layout=widgets.Layout(width='100%'),
    )
)

In [ ]:
histograma_ndvi = (
    ndvi.reduceRegion(
        reducer=ee.Reducer.histogram(maxBuckets=50),
        geometry=aoi,
        scale=10,
        maxPixels=1e7,
    )
    .get('NDVI')
    .getInfo()
)

estadisticas_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean().combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True,
    ),
    geometry=aoi,
    scale=10,
    maxPixels=1e7,
).getInfo()

media_ndvi = estadisticas_ndvi['NDVI_mean']
mediana_ndvi = estadisticas_ndvi['NDVI_median']

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    histograma_ndvi['bucketMeans'],
    histograma_ndvi['histogram'],
    width=0.02,
    color=COLORES_INDICES['NDVI'],
    alpha=0.85,
)
ax.axvline(media_ndvi, color='#e76f51', linestyle='--', label=f'Media: {media_ndvi:.2f}')
ax.axvline(mediana_ndvi, color='#264653', linestyle=':', label=f'Mediana: {mediana_ndvi:.2f}')
ax.set(
    xlabel='NDVI',
    ylabel='Cantidad de píxeles',
    title='Distribución espacial del NDVI en la parcela',
)
ax.legend()
plt.show()

## 8 Serie temporal de NDVI

Cada imagen se convierte en una observación con fecha y NDVI medio. La línea conecta fechas disponibles, pero no implica que existan mediciones diarias.

**Qué observar:** identifique máximos, mínimos, cambios rápidos y vacíos temporales. Una trayectoria gradual puede ser compatible con fenología; un salto aislado requiere revisar nubes, sombras y cantidad de píxeles válidos.

In [ ]:
def resumir_ndvi(imagen):
    # Resumir el NDVI medio del área para una imagen Sentinel-2.
    ndvi_imagen = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndvi_medio = ndvi_imagen.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10,
        maxPixels=1e7,
    ).get('NDVI')

    return ee.Feature(None, {
        'fecha': imagen.date().format('YYYY-MM-dd'),
        'tiempo': imagen.get('system:time_start'),
        'NDVI': ndvi_medio,
    })


# Reconstruir la colección temporal con las fechas actualmente configuradas.
# Así esta serie no reutiliza una colección creada con fechas anteriores.
s2_temporal_ndvi = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', NUBOSIDAD_MAX))
    .sort('system:time_start')
    .map(preparar_s2)
)

lista_imagenes_ndvi = s2_temporal_ndvi.toList(s2_temporal_ndvi.size())
lista_ndvi = lista_imagenes_ndvi.map(
    lambda elemento: resumir_ndvi(ee.Image(elemento))
)

serie_ndvi_ee = (
    ee.FeatureCollection(lista_ndvi)
    .filter(ee.Filter.notNull(['NDVI']))
    .sort('tiempo')
)

registros_ndvi = [
    elemento['properties']
    for elemento in serie_ndvi_ee.getInfo()['features']
]

if not registros_ndvi:
    raise ValueError(
        'No hay observaciones válidas de NDVI para el periodo seleccionado.'
    )

serie_ndvi = pd.DataFrame(registros_ndvi)
serie_ndvi['fecha'] = pd.to_datetime(serie_ndvi['fecha'])
serie_ndvi['NDVI'] = pd.to_numeric(serie_ndvi['NDVI'], errors='coerce')
serie_ndvi = serie_ndvi.sort_values('fecha').reset_index(drop=True)

print('Periodo configurado:', FECHA_INICIO, 'a', FECHA_FIN)
print('Primera observación NDVI:', serie_ndvi['fecha'].min().date())
print('Última observación NDVI:', serie_ndvi['fecha'].max().date())
print('Observaciones válidas:', len(serie_ndvi))
display(serie_ndvi[['fecha', 'NDVI']].head())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    serie_ndvi['fecha'],
    serie_ndvi['NDVI'],
    marker='o',
    markersize=4,
    color=COLORES_INDICES['NDVI'],
    linewidth=1.5,
)

# Mostrar todo el intervalo solicitado, incluso si existen vacíos sin imágenes.
ax.set_xlim(
    pd.Timestamp(FECHA_INICIO),
    pd.Timestamp(FECHA_FIN),
)
ax.set_ylim(0, 1)
ax.set_xlabel('Fecha de adquisición')
ax.set_ylabel('NDVI medio')
ax.set_title(
    f'Evolución temporal del NDVI: {FECHA_INICIO} a {FECHA_FIN}'
)

fig.autofmt_xdate()
plt.show()


### Punto de control 3

Describa la forma de la serie sin explicar todavía su causa. Identifique máximos, mínimos, vacíos y cambios rápidos. Después proponga una hipótesis fenológica y una hipótesis relacionada con estrés.

## 9 Índices de vigor, humedad y posible estrés

Los índices resumen respuestas espectrales diferentes. Sus rangos deben interpretarse de forma relativa para la misma cobertura, estación y procesamiento.

| Índice | Bandas | Para qué sirve | Lectura orientativa |
|---|---|---|---|
| **NDVI** | B8 y B4 | Actividad y cobertura de vegetación verde | Mayor suele indicar más cobertura o vigor; puede saturarse en doseles densos. |
| **NDMI** | B8 y B11 | Humedad del dosel y respuesta del agua en la vegetación | Mayor suele asociarse con mayor contenido de humedad; menor con condiciones más secas. |
| **MSI** | B11/B8 | Sensibilidad al estrés por humedad | Mayor suele acompañar menor contenido de agua; su dirección tiende a ser inversa a NDMI. |
| **NMDI** | B8, B11 y B12 | Cambios combinados en humedad de suelo y vegetación | Su respuesta depende de la etapa fenológica y del suelo; conviene interpretar su trayectoria, no un umbral único. |
| **DWSI** | B8, B3, B11 y B4 | Respuesta conjunta asociada con enfermedad y estrés hídrico | Aumentos pueden acompañar mayor demanda o estrés, pero también variación estacional del dosel. |

En Zambrano et al. (2025), los índices con SWIR —especialmente MSI, DWSI, NDMI y NMDI— estuvieron entre los predictores satelitales más relevantes. La meteorología tuvo mayor peso en el comportamiento temporal, por lo que ningún índice debe interpretarse como diagnóstico aislado.

In [ ]:
def calcular_indices(imagen):
    # Agregar cinco índices espectrales con nombres estandarizados.
    ndvi_imagen = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi_imagen = imagen.normalizedDifference(['B8', 'B11']).rename('NDMI')
    msi_imagen = (
        imagen.select('B11')
        .divide(imagen.select('B8'))
        .rename('MSI')
    )
    nmdi_imagen = (
        imagen.select('B8')
        .subtract(imagen.select('B11').subtract(imagen.select('B12')))
        .divide(
            imagen.select('B8')
            .add(imagen.select('B11').subtract(imagen.select('B12')))
        )
        .rename('NMDI')
    )
    dwsi_imagen = (
        imagen.select('B8').add(imagen.select('B3'))
        .divide(imagen.select('B11').add(imagen.select('B4')))
        .rename('DWSI')
    )

    bandas_adicionales = ee.Image([
        ndmi_imagen,
        msi_imagen,
        nmdi_imagen,
        dwsi_imagen,
    ])
    indices = ndvi_imagen.addBands(bandas_adicionales)

    # copyProperties retorna ee.Element; ee.Image recupera el tipo concreto.
    return ee.Image(
        indices.copyProperties(imagen, ['system:time_start'])
    )


indices_compuesto = calcular_indices(compuesto)
NOMBRES_INDICES = ['NDVI', 'NDMI', 'MSI', 'NMDI', 'DWSI']

VIS_INDICES = {
    'NDVI': VIS_NDVI,
    'NDMI': {'min': -0.2, 'max': 0.6, 'palette': ['8c510a', 'f7f7f7', '2166ac']},
    'MSI': {'min': 0.4, 'max': 1.5, 'palette': ['1a9850', 'fee08b', 'd73027']},
    'NMDI': {'min': 0.3, 'max': 1.0, 'palette': ['d73027', 'fee08b', '1a9850']},
    'DWSI': {'min': 0.5, 'max': 2.0, 'palette': ['2166ac', 'f7f7f7', 'b2182b']},
}

# Deje activa solamente una línea para cambiar el índice mostrado.
INDICE_MAPA = 'NDMI'
# INDICE_MAPA = 'MSI'
# INDICE_MAPA = 'NMDI'
# INDICE_MAPA = 'DWSI'
# INDICE_MAPA = 'NDVI'

if INDICE_MAPA not in NOMBRES_INDICES:
    raise ValueError(f'Índice no reconocido: {INDICE_MAPA}')

mapa_indice = geemap.Map(basemap={})
mapa_indice.centerObject(aoi, 15)
mapa_indice.addLayer(
    indices_compuesto.select(INDICE_MAPA),
    VIS_INDICES[INDICE_MAPA],
    INDICE_MAPA,
)
mapa_indice.addLayer(
    aoi_fc.style(**{'color': 'black', 'fillColor': '00000000', 'width': 2}),
    {},
    'Parcela',
)

print('Índice mostrado:', INDICE_MAPA)
mapa_indice

### Guardar y comparar categorías

La celda siguiente guarda la media de los cinco índices con el nombre de la categoría actual. Para comparar otra especie o uso del suelo:

1. vuelva a las líneas comentadas de `CATEGORIA_ANALISIS`;
2. active otra categoría y comente la anterior;
3. ejecute nuevamente desde la validación de la categoría;
4. vuelva a ejecutar la celda de comparación.

La tabla conservará las categorías analizadas durante la sesión de Colab.

In [ ]:
estadisticas_categoria = indices_compuesto.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi,
    scale=20,
    maxPixels=1e7,
).getInfo()

try:
    resultados_comparacion
except NameError:
    resultados_comparacion = {}

clave_resultado = f'{TIPO_ANALISIS} | {CATEGORIA_ANALISIS}'
resultados_comparacion[clave_resultado] = estadisticas_categoria

tabla_comparacion = (
    pd.DataFrame(resultados_comparacion)
    .T[NOMBRES_INDICES]
    .sort_index()
)

display(tabla_comparacion.round(3))

## 10 Análisis temporal conjunto de los índices

Ahora se calculan los cinco índices para cada fecha. Se mantiene una observación por imagen y se resume cada índice mediante la media de la parcela.

**Qué observar:** compare tendencias, fechas de máximos y mínimos y cambios que aparecen simultáneamente. NDMI y MSI suelen moverse en direcciones opuestas. NMDI y DWSI requieren especial atención al contexto fenológico señalado en la tabla anterior.

In [ ]:
def resumir_indices(imagen):
    # Calcular la media espacial de todos los índices para una fecha.
    estadisticas = calcular_indices(imagen).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=20,
        maxPixels=1e7,
    )

    resumen = ee.Feature(None, estadisticas).set({
        'fecha': imagen.date().format('YYYY-MM-dd'),
        'tiempo': imagen.get('system:time_start'),
    })
    return ee.Feature(resumen)


# Crear la lista aquí para que esta sección no dependa del punto 8.
lista_imagenes_indices = s2.toList(s2.size())
lista_indices = lista_imagenes_indices.map(
    lambda elemento: resumir_indices(ee.Image(elemento))
)

serie_indices_ee = (
    ee.FeatureCollection(lista_indices)
    .filter(ee.Filter.notNull(NOMBRES_INDICES))
    .sort('tiempo')
)

registros_indices = [
    elemento['properties']
    for elemento in serie_indices_ee.getInfo()['features']
]

if not registros_indices:
    raise ValueError(
        'No hay fechas con datos válidos para todos los índices. '
        'Amplíe el periodo o aumente NUBOSIDAD_MAX.'
    )

serie_indices = pd.DataFrame(registros_indices)
serie_indices['fecha'] = pd.to_datetime(serie_indices['fecha'])
serie_indices = serie_indices.sort_values('fecha').reset_index(drop=True)

for indice in NOMBRES_INDICES:
    serie_indices[indice] = pd.to_numeric(
        serie_indices[indice],
        errors='coerce',
    )

print('Fechas válidas para todos los índices:', len(serie_indices))
display(serie_indices[['fecha', *NOMBRES_INDICES]].head())

In [ ]:
fig, ejes = plt.subplots(
    1,
    2,
    figsize=(15, 4.5),
    constrained_layout=True,
)

for indice in ['NDVI', 'NDMI', 'NMDI']:
    ejes[0].plot(
        serie_indices['fecha'],
        serie_indices[indice],
        marker='o',
        linewidth=1.5,
        label=indice,
        color=COLORES_INDICES[indice],
    )

ejes[0].set(
    xlabel='Fecha',
    ylabel='Valor medio',
    title='Índices normalizados',
)
ejes[0].legend()

for indice in ['MSI', 'DWSI']:
    ejes[1].plot(
        serie_indices['fecha'],
        serie_indices[indice],
        marker='o',
        linewidth=1.5,
        label=indice,
        color=COLORES_INDICES[indice],
    )

ejes[1].set(
    xlabel='Fecha',
    ylabel='Razón espectral media',
    title='Índices de humedad y posible estrés',
)
ejes[1].legend()

for eje in ejes:
    eje.tick_params(axis='x', rotation=35)

plt.show()

### Resumen de la dinámica temporal

La tabla siguiente ubica el mínimo, el máximo y la amplitud observada de cada índice. La amplitud permite comparar cuánto varió cada indicador durante el periodo, pero no debe compararse directamente entre índices con escalas diferentes.

In [ ]:
resumen_temporal = []

for indice in NOMBRES_INDICES:
    serie_valida = serie_indices[['fecha', indice]].dropna()
    fila_minima = serie_valida.loc[serie_valida[indice].idxmin()]
    fila_maxima = serie_valida.loc[serie_valida[indice].idxmax()]

    resumen_temporal.append({
        'Índice': indice,
        'Mínimo': fila_minima[indice],
        'Fecha mínimo': fila_minima['fecha'].date(),
        'Máximo': fila_maxima[indice],
        'Fecha máximo': fila_maxima['fecha'].date(),
        'Amplitud': fila_maxima[indice] - fila_minima[indice],
    })

tabla_temporal = pd.DataFrame(resumen_temporal).set_index('Índice')
display(tabla_temporal.round(3))

## 11 Correlación entre índices

La correlación de Spearman evalúa si dos índices cambian de manera monotónica, aunque sus escalas sean distintas.

- valores próximos a **+1** indican que ambos tienden a aumentar juntos;
- valores próximos a **−1** indican comportamientos opuestos;
- valores próximos a **0** indican poca relación monotónica;
- una correlación alta puede señalar información redundante o una estacionalidad compartida;
- correlación no implica causalidad ni confirma estrés.

**Qué observar:** compruebe si NDMI y MSI muestran relación negativa y localice pares con correlación absoluta alta. Después compare esos resultados con las formas de las series temporales.

In [ ]:
matriz_correlacion = serie_indices[NOMBRES_INDICES].corr(
    method='spearman'
)

fig, ax = plt.subplots(figsize=(7, 6))
imagen_correlacion = ax.imshow(
    matriz_correlacion,
    cmap='RdBu_r',
    vmin=-1,
    vmax=1,
)

ax.set_xticks(range(len(NOMBRES_INDICES)), NOMBRES_INDICES, rotation=45, ha='right')
ax.set_yticks(range(len(NOMBRES_INDICES)), NOMBRES_INDICES)
ax.set_title('Correlación de Spearman entre índices')
ax.grid(False)

for fila in range(len(NOMBRES_INDICES)):
    for columna in range(len(NOMBRES_INDICES)):
        valor = matriz_correlacion.iloc[fila, columna]
        color_texto = 'white' if abs(valor) >= 0.60 else 'black'
        ax.text(
            columna,
            fila,
            f'{valor:.2f}',
            ha='center',
            va='center',
            color=color_texto,
        )

fig.colorbar(
    imagen_correlacion,
    ax=ax,
    label='Coeficiente de Spearman',
    shrink=0.82,
)
plt.show()

# Mostrar los pares más relacionados sin repetir la diagonal.
triangulo_superior = np.triu(
    np.ones(matriz_correlacion.shape, dtype=bool),
    k=1,
)

pares_correlacion = (
    matriz_correlacion
    .where(triangulo_superior)
    .stack()
    .rename('rho')
    .reset_index()
    .rename(columns={'level_0': 'Índice 1', 'level_1': 'Índice 2'})
)
pares_correlacion['|rho|'] = pares_correlacion['rho'].abs()

display(
    pares_correlacion
    .sort_values('|rho|', ascending=False)
    .head(10)
    .round(3)
)

### Lectura integrada de los resultados

1. **Fenología:** busque cambios graduales y coherentes entre fechas. En cerezos, la expansión foliar y el periodo de mayor actividad pueden elevar NDVI y NDMI.
2. **Humedad:** una disminución persistente de NDMI acompañada por un aumento de MSI merece atención, especialmente si coincide con alta demanda atmosférica.
3. **Posible estrés:** cambios en DWSI, MSI o NMDI deben contrastarse con temperatura, déficit de presión de vapor, evapotranspiración, riego y observaciones de campo.
4. **Degradación:** una caída persistente de índices de vegetación puede ser compatible con pérdida de cobertura, pero también con cosecha, poda, senescencia, suelo visible o cambios de manejo.
5. **Correlación:** índices muy correlacionados pueden describir el mismo ciclo estacional. Para explicar procesos o construir modelos, conviene combinar información espectral con variables meteorológicas y mediciones de terreno.

En Zambrano et al. (2025), la meteorología explicó mejor la variación temporal del potencial hídrico del tallo, mientras que los índices con SWIR aportaron información importante sobre su variación espacial.

## Cómo interpretar sin exceder la evidencia

- Describa primero el patrón observado y separe esa descripción de su explicación.
- Una variación estacional repetida puede ser compatible con fenología.
- Un cambio breve debe revisarse contra nubes, sombras, resolución espacial y cantidad de observaciones.
- NDVI resume actividad y cobertura verde, pero puede saturarse en vegetación densa.
- Los índices con SWIR añaden sensibilidad al contenido de agua y a la estructura del dosel.
- Los rangos dependen de especie, suelo, arquitectura, manejo, fecha y procesamiento; no aplique umbrales universales.
- Ningún índice confirma por sí solo estrés hídrico, enfermedad o degradación.
- Una correlación alta entre índices no demuestra que uno cause al otro.
- Zambrano et al. combinaron Sentinel-2, meteorología, potencial hídrico medido y aprendizaje automático.

## Fuentes para continuar

- Zambrano, F. et al. (2025). *Prediction of the daily spatial variation of stem water potential in cherry orchards using weather and Sentinel-2 data*. Agricultural Water Management, 318, 109721. https://doi.org/10.1016/j.agwat.2025.109721
- Google Earth Engine Python quickstart. https://developers.google.com/earth-engine/guides/quickstart_python
- Autenticación e inicialización de Earth Engine. https://developers.google.com/earth-engine/guides/auth
- Sentinel-2 SR Harmonized. https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Sentinel-2 Harmonized TOA. https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
- Documentación de geemap. https://geemap.org/